# 01 — Análisis exploratorio y detección de duplicados

**Qué hace:** explora el dataset original (`asl_dataset/`): cuenta clases e imágenes, revisa dimensiones/formato/canales, calcula estadísticas de píxeles (media y desviación por canal) y detecta anomalías (imágenes negras, blancas, planas o corruptas). Luego busca duplicados exactos (hash MD5) tanto en `asl_dataset/` como en `asl_processed/`, y finalmente agrupa near-duplicates (fotos de la misma toma) usando pHash.

**Consume:** `asl_dataset/<clase>/*` (imágenes originales), `asl_processed/<split>/<clase>/*` (imágenes ya repartidas en train/test).

**Produce:** solo salidas impresas/gráficas de diagnóstico — no escribe archivos. Sirve de base para las decisiones que se aplican en `02_split_por_grupos.ipynb`.

## Imports y rutas base

In [3]:
from pathlib import Path
from PIL import Image
from collections import Counter
import numpy as np
import imagehash

In [8]:
DATA_DIR = Path('./asl_dataset')

## Conteo de clases e imágenes

In [60]:
classes = sorted([folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()])
print(f'número de clases: {len(classes)}, {classes}')

número de clases: 36, ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [4]:
counting = { class_name: len(list((DATA_DIR / class_name).glob('*'))) for class_name in classes}
print(f'número de imágenes por clase: {counting}')


número de imágenes por clase: {'0': 1000, '1': 1000, '2': 1000, '3': 1000, '4': 1000, '5': 1000, '6': 1000, '7': 1000, '8': 1000, '9': 1000, 'A': 1000, 'B': 1000, 'C': 1000, 'D': 1000, 'E': 1000, 'F': 1000, 'G': 1000, 'H': 1000, 'I': 1000, 'J': 1000, 'K': 1000, 'L': 1000, 'M': 1000, 'N': 1000, 'O': 1000, 'P': 1000, 'Q': 1000, 'R': 1000, 'S': 1000, 'T': 1000, 'U': 1000, 'V': 1000, 'W': 1000, 'X': 1000, 'Y': 1000, 'Z': 1000}


## Dimensiones, formato y canales de las imágenes

In [5]:
#dimension, formato, tamaño, canales

size, mode, formats = Counter(), Counter(), Counter()

for class_ in classes:
    for file_ in list((DATA_DIR / class_).glob('*')):
        with Image.open(file_) as img:
            size[img.size] += 1
            mode[img.mode] += 1
            formats[file_.suffix] += 1

print(f'tamaños {size}')
print(f'modos de color {mode}')
print(f'formatos {formats}')

tamaños Counter({(300, 300): 36000})
modos de color Counter({'RGB': 36000})
formatos Counter({'.jpg': 36000})


In [6]:

muestra = np.array(Image.open(next((DATA_DIR / classes[0]).glob('*'))))
print("Shape:", muestra.shape, "| min:", muestra.min(),
      "| max:", muestra.max(), "| dtype:", muestra.dtype)

Shape: (300, 300, 3) | min: 0 | max: 250 | dtype: uint8


## Estadísticas de píxeles (media y desviación estándar por canal)

In [11]:
## analisis de pixel

count = 0
sum = np.zeros(3, dtype=np.float64)
sum_square = np.zeros(3, dtype=np.float64)


for file_ in list((DATA_DIR / class_).glob('*')):
    with Image.open(file_) as img:
        xvalue = np.asanyarray(img, dtype=np.float64) / 255

    sum += xvalue.sum(axis=(0,1))
    sum_square += (xvalue ** 2).sum(axis=(0,1))
    count += xvalue.shape[0] * xvalue.shape[1]

mean = sum / count
standard_deviation = np.sqrt(sum_square / count - mean ** 2)


print(f'media: {mean}')
print(f'standard deviation {standard_deviation}')

media: [0.77403883 0.73730229 0.71300213]
standard deviation [0.15953144 0.19355108 0.20650022]


## Detección de anomalías: imágenes negras, blancas, planas o corruptas

In [19]:
black_umbral = 0.02
white_umbral = 0.98
plain_umbral  = 0.01


corrupt = []
black = []
white = []
plain = []

for file_ in list((DATA_DIR / class_).glob('*')):
    try:
        with Image.open(file_) as img:
            x = np.asanyarray(img, dtype=np.float64) / 255.0
    except:
        corrupt.append(file_)
    mean = x.mean()
    desv = x.std()

    if mean < black_umbral:
        black.append((file_, mean))
    elif mean > white_umbral:
        white.append((file_, mean))
    elif desv < plain_umbral:      # 'elif': una plana no-negra/no-blanca
        plain.append((file_, desv))


print(f'black: {black}')
print(f'white: {white}')
print(f'plain: {plain}')
print(f'corrupt: {corrupt}')


black: []
white: []
plain: []
corrupt: []


## Duplicados exactos (MD5) en `asl_dataset/`

In [41]:
import hashlib
from collections import defaultdict

## identificar duplicados

# Diccionario: huella → lista de archivos que la tienen.
# defaultdict(list) crea automáticamente una lista vacía para cada huella nueva.
hash_a_archivos = defaultdict(list)

archivos = list(DATA_DIR.glob("*/*"))
print(f"Calculando huellas de {len(archivos)} archivos...\n")


for ruta in archivos:
    # Leemos el archivo como bytes crudos (no como imagen) y sacamos su MD5
    contenido = ruta.read_bytes()
    huella = hashlib.md5(contenido).hexdigest()
    hash_a_archivos[huella].append(ruta)


# Un grupo con más de un archivo = duplicados exactos
grupos_duplicados = {h: rutas for h, rutas in hash_a_archivos.items() if len(rutas) > 1}

# --- Reporte ---
total_unicas = len(hash_a_archivos)        # nº de huellas distintas = imágenes únicas
total_archivos = len(archivos)
total_sobrantes = total_archivos - total_unicas  # cuántas borrarías


print(f"Archivos totales:     {total_archivos}")
print(f"Imágenes únicas:      {total_unicas}")
print(f"Grupos con duplicados:{len(grupos_duplicados)}")
print(f"Archivos sobrantes:   {total_sobrantes}  (los que eliminarías)\n")


Calculando huellas de 36000 archivos...

Archivos totales:     36000
Imágenes únicas:      35441
Grupos con duplicados:435
Archivos sobrantes:   559  (los que eliminarías)



In [27]:
# Mostramos algunos grupos para inspeccionar
for huella, rutas in list(grupos_duplicados.items())[:5]:
    print(f"Huella {huella[:8]}... aparece {len(rutas)} veces:")
    for r in rutas:
        print(f"   {r.parent.name}/{r.name}")
    print()

Huella 82c8465f... aparece 3 veces:
   R/P10_R_928.jpg
   R/P10_R_927.jpg
   R/P10_R_926.jpg

Huella 35140c68... aparece 3 veces:
   R/P2_R_175.jpg
   R/P2_R_174.jpg
   R/P2_R_173.jpg

Huella eb425a29... aparece 2 veces:
   R/P5_R_448.jpg
   R/P5_R_447.jpg

Huella ff49b359... aparece 7 veces:
   R/P5_R_438.jpg
   R/P5_R_439.jpg
   R/P5_R_443.jpg
   R/P5_R_442.jpg
   R/P5_R_440.jpg
   R/P5_R_441.jpg
   R/P5_R_444.jpg

Huella 4c7bd658... aparece 2 veces:
   R/P2_R_134.jpg
   R/P2_R_136.jpg



## Near-duplicates (pHash) dentro de `asl_dataset/`

In [ ]:
UMBRAL_HAMMING = 5   # distancia <= 5 => las consideramos "el mismo grupo visual"
clases = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
resumen = {}   # clase -> nº de grupos visuales únicos

In [4]:

for clase in clases:
    archivos = sorted((DATA_DIR / clase).glob("*"))

    # 1. Calcular el pHash de cada imagen de la clase
    hashes = []
    for ruta in archivos:
        with Image.open(ruta) as img:
            hashes.append((ruta, imagehash.phash(img)))

    # 2. Agrupar por cercanía (estrategia "greedy": comparar contra representantes)
    grupos = []   # cada grupo: {'rep': hash_representante, 'archivos': [...]}
    for ruta, h in hashes:
        colocada = False
        for g in grupos:
            if h - g['rep'] <= UMBRAL_HAMMING:   # '-' entre hashes = distancia de Hamming
                g['archivos'].append(ruta)
                colocada = True
                break
        if not colocada:
            grupos.append({'rep': h, 'archivos': [ruta]})

    resumen[clase] = len(grupos)
    print(f"  {clase}: {len(grupos)} grupos visuales")

total = sum(resumen.values())
print(f"\nTotal de grupos visuales en el dataset: {total}")
print(f"(si tu teoría de '10 fotos-madre por clase' es cierta, esperarías ~360)")

  0: 85 grupos visuales
  1: 50 grupos visuales
  2: 36 grupos visuales
  3: 50 grupos visuales
  4: 55 grupos visuales
  5: 45 grupos visuales
  6: 37 grupos visuales
  7: 53 grupos visuales
  8: 45 grupos visuales
  9: 42 grupos visuales
  A: 72 grupos visuales
  B: 54 grupos visuales
  C: 47 grupos visuales
  D: 70 grupos visuales
  E: 54 grupos visuales
  F: 48 grupos visuales
  G: 54 grupos visuales
  H: 66 grupos visuales
  I: 47 grupos visuales
  J: 48 grupos visuales
  K: 43 grupos visuales
  L: 45 grupos visuales
  M: 72 grupos visuales
  N: 49 grupos visuales
  O: 49 grupos visuales
  P: 54 grupos visuales
  Q: 47 grupos visuales
  R: 53 grupos visuales
  S: 55 grupos visuales
  T: 55 grupos visuales
  U: 52 grupos visuales
  V: 44 grupos visuales
  W: 62 grupos visuales
  X: 64 grupos visuales
  Y: 64 grupos visuales
  Z: 47 grupos visuales

Total de grupos visuales en el dataset: 1913
(si tu teoría de '10 fotos-madre por clase' es cierta, esperarías ~360)


## Duplicados exactos (MD5) en `asl_processed/`

CHECK DUPLICATES IN PROCESSED

In [2]:
DATA_DIR_PROCESSED = Path('./asl_processed')

In [3]:
import hashlib
from collections import defaultdict

## identificar duplicados

# Diccionario: huella → lista de archivos que la tienen.
# defaultdict(list) crea automáticamente una lista vacía para cada huella nueva.
hash_a_archivos = defaultdict(list)

archivos = list(DATA_DIR_PROCESSED.glob("*/*/*"))
## archivos = [p for p in DATA_DIR_PROCESSED.rglob("*") if p.is_file()]
print(f"Calculando huellas de {len(archivos)} archivos...\n")


for ruta in archivos:
    # Leemos el archivo como bytes crudos (no como imagen) y sacamos su MD5
    contenido = ruta.read_bytes()
    huella = hashlib.md5(contenido).hexdigest()
    hash_a_archivos[huella].append(ruta)


# Un grupo con más de un archivo = duplicados exactos
grupos_duplicados = {h: rutas for h, rutas in hash_a_archivos.items() if len(rutas) > 1}

# --- Reporte ---
total_unicas = len(hash_a_archivos)        # nº de huellas distintas = imágenes únicas
total_archivos = len(archivos)
total_sobrantes = total_archivos - total_unicas  # cuántas borrarías


print(f"Archivos totales:     {total_archivos}")
print(f"Imágenes únicas:      {total_unicas}")
print(f"Grupos con duplicados:{len(grupos_duplicados)}")
print(f"Archivos sobrantes:   {total_sobrantes}  (los que eliminarías)\n")

Calculando huellas de 35404 archivos...

Archivos totales:     35404
Imágenes únicas:      35404
Grupos con duplicados:0
Archivos sobrantes:   0  (los que eliminarías)



In [44]:
# Mostramos algunos grupos para inspeccionar
for huella, rutas in list(grupos_duplicados.items())[:5]:
    print(f"Huella {huella[:8]}... aparece {len(rutas)} veces:")
    for r in rutas:
        print(f"{r.parent.parent.name}/{r.parent.name}/{r.name}")
    print()

## Near-duplicates (pHash) en `asl_processed/` (train + test combinados)

NEAR DUPLICATE

In [45]:
DATA_DIR_PROCESSED = Path('./asl_processed')
UMBRAL_HAMMING = 5

# clases desde train (o test; deberían ser las mismas)
clases = sorted(d.name for d in (DATA_DIR_PROCESSED / "test").iterdir() if d.is_dir())
resumen = {}

for clase in clases:
    # todas las imágenes de esa clase en train y test
    archivos = sorted(DATA_DIR_PROCESSED.glob(f"*/{clase}/*"))
    archivos = [p for p in archivos if p.is_file()]  # evita carpetas/.DS_Store

    hashes = []
    for ruta in archivos:
        with Image.open(ruta) as img:
            hashes.append((ruta, imagehash.phash(img)))

    grupos = []
    for ruta, h in hashes:
        colocada = False
        for g in grupos:
            if h - g['rep'] <= UMBRAL_HAMMING:
                g['archivos'].append(ruta)
                colocada = True
                break
        if not colocada:
            grupos.append({'rep': h, 'archivos': [ruta]})

    resumen[clase] = len(grupos)
    print(f"  {clase}: {len(grupos)} grupos visuales ({len(archivos)} imágenes)")

total = sum(resumen.values())
print(f"\nTotal de grupos visuales: {total}")

  0: 213 grupos visuales (975 imágenes)
  1: 41 grupos visuales (991 imágenes)
  2: 39 grupos visuales (992 imágenes)
  3: 34 grupos visuales (998 imágenes)
  4: 41 grupos visuales (997 imágenes)
  5: 25 grupos visuales (972 imágenes)
  6: 38 grupos visuales (991 imágenes)
  7: 35 grupos visuales (995 imágenes)
  8: 54 grupos visuales (993 imágenes)
  9: 37 grupos visuales (976 imágenes)
  A: 31 grupos visuales (986 imágenes)
  B: 34 grupos visuales (990 imágenes)
  C: 34 grupos visuales (986 imágenes)
  D: 49 grupos visuales (992 imágenes)
  E: 26 grupos visuales (988 imágenes)
  F: 32 grupos visuales (1000 imágenes)
  G: 25 grupos visuales (985 imágenes)
  H: 47 grupos visuales (978 imágenes)
  I: 31 grupos visuales (893 imágenes)
  J: 30 grupos visuales (995 imágenes)
  K: 38 grupos visuales (981 imágenes)
  L: 30 grupos visuales (977 imágenes)
  M: 58 grupos visuales (994 imágenes)
  N: 35 grupos visuales (979 imágenes)
  O: 35 grupos visuales (995 imágenes)
  P: 60 grupos visuales